# Bank Marketing Term Deposit Prediction

**CLD 375 DE-ML Final Project.** Section numbers follow the project document template.

## 0. Setup

In [ ]:
import sys
import subprocess
import urllib.request
import zipfile
from pathlib import Path

REPO_URL = "https://github.com/Soxavin/Cloud-ML-DE-Final-Project.git"
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # Colab starts empty every session, so grab our repo (it has the shared bankml package)
    REPO_DIR = Path("/content/Cloud-ML-DE-Final-Project")
    if REPO_DIR.exists():
        subprocess.run(["git", "-C", str(REPO_DIR), "pull"], check=True)
    else:
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
else:
    # running locally from the notebooks/ folder of the repo
    REPO_DIR = Path.cwd().parent

sys.path.insert(0, str(REPO_DIR))
print("Repo folder:", REPO_DIR)

In [ ]:
import platform
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import scipy
import sklearn
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline

from bankml.pipeline import build_preprocessor

# scikit-learn 1.6.1 (Colab's version) passes an outdated "iprint" option to SciPy's optimiser, and newer
# SciPy warns about it on every logistic regression fit. Harmless, so we hide only that one message.
warnings.filterwarnings("ignore", message=".*iprint.*")

RANDOM_STATE = 42
# one CV splitter shared by every experiment, so all models see exactly the same folds
CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 30)

# these exact versions get pinned in the model service later, so the saved model loads the same way
VERSIONS = {
    "python": platform.python_version(),
    "scikit-learn": sklearn.__version__,
    "scipy": scipy.__version__,
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "joblib": joblib.__version__,
}
VERSIONS

## 1. Introduction

A Portuguese bank ran phone campaigns between May 2008 and November 2010 to sell term deposits. Most calls
fail: only about 1 in 9 clients said yes. If the bank could tell beforehand which clients are likely to
say yes, its call centre could call those first and waste fewer calls.

We use the UCI Bank Marketing dataset (`bank-additional-full.csv`, Moro et al., 2014): 41,188 calls with
client details, campaign details and the economic situation at the time of each call.

## 2. Objective

1. Predict whether a client will subscribe (`y` = yes/no) using only information known **before** the call.
2. Clean the data and choose the input features, explaining every decision.
3. Train and compare 7 classification models, then combine the best 3 with a meta-classifier (stacking).
4. Save the final model so it can be deployed behind an API on AWS EC2.

## 3. Dataset and column details

### 3.1 Load the data

In [ ]:
UCI_URL = "https://archive.ics.uci.edu/static/public/222/bank+marketing.zip"
DATA_DIR = REPO_DIR / "data"


def find_file(folder, name):
    """Find a file anywhere under folder, skipping the __MACOSX junk that zips from Macs carry."""
    for path in folder.rglob(name):
        if "__MACOSX" not in path.parts:
            return path
    return None


csv_path = find_file(DATA_DIR, "bank-additional-full.csv") if DATA_DIR.exists() else None

if csv_path is None:
    # the CSV sits in a zip inside the zip, so we unzip twice
    DATA_DIR.mkdir(exist_ok=True)
    outer_zip = DATA_DIR / "bank_marketing.zip"
    urllib.request.urlretrieve(UCI_URL, outer_zip)
    with zipfile.ZipFile(outer_zip) as z:
        z.extractall(DATA_DIR)
    with zipfile.ZipFile(find_file(DATA_DIR, "bank-additional.zip")) as z:
        z.extractall(DATA_DIR)
    csv_path = find_file(DATA_DIR, "bank-additional-full.csv")

# the file uses ; as the separator, not commas
df_raw = pd.read_csv(csv_path, sep=";")
print(csv_path)
df_raw.head()

### 3.2 Check the raw data

These numbers come from our first profile of the file. If any line fails,
stop and look before continuing, because everything after this depends on them.

In [ ]:
assert df_raw.shape == (41188, 21)
assert df_raw.isna().sum().sum() == 0
assert df_raw.duplicated().sum() == 12
assert (df_raw["default"] == "unknown").sum() == 8597
assert round((df_raw["pdays"] == 999).mean(), 3) == 0.963
assert ((df_raw["previous"] > 0) & (df_raw["pdays"] == 999)).sum() == 4110
assert round((df_raw["y"] == "yes").mean(), 3) == 0.113
print("All checks passed")

### 3.3 Column details

In [ ]:
descriptions = {
    "age": "Client age in years",
    "job": "Type of job",
    "marital": "Marital status (divorced includes widowed)",
    "education": "Education level",
    "default": "Has credit in default?",
    "housing": "Has a housing loan?",
    "loan": "Has a personal loan?",
    "contact": "Contact type: cellular or telephone",
    "month": "Month of the last contact",
    "day_of_week": "Weekday of the last contact",
    "duration": "Length of the last call in seconds (only known after the call)",
    "campaign": "Number of calls to this client in this campaign, including the last one",
    "pdays": "Days since last contact in a previous campaign (999 = never contacted)",
    "previous": "Number of contacts before this campaign",
    "poutcome": "Outcome of the previous campaign",
    "emp.var.rate": "Employment variation rate (quarterly, economy-wide)",
    "cons.price.idx": "Consumer price index (monthly, economy-wide)",
    "cons.conf.idx": "Consumer confidence index (monthly, economy-wide)",
    "euribor3m": "3-month Euribor interest rate (daily, economy-wide)",
    "nr.employed": "Number of employees in the economy (quarterly)",
    "y": "TARGET: did the client subscribe to a term deposit?",
}

column_details = pd.DataFrame({
    "column": df_raw.columns,
    "type": df_raw.dtypes.astype(str).values,
    "unique_values": df_raw.nunique().values,
    "example": df_raw.iloc[0].values,
    "description": df_raw.columns.map(descriptions),
})
column_details

The 20 inputs fall into three groups: **client details** (age to loan), **campaign details**
(contact to poutcome) and **economic context** (the last five, the same for everyone called in the
same period). `y` is the target.

## 4. Data cleaning and preprocessing

### 4.1 Missing values

In [ ]:
print("Real missing values (NaN):", df_raw.isna().sum().sum())

categorical_cols = list(df_raw.select_dtypes(exclude="number").columns)
unknown_counts = (
    (df_raw[categorical_cols] == "unknown")
    .sum()
    .sort_values(ascending=False)
)
unknown_counts = unknown_counts[unknown_counts > 0]
unknown_counts

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
unknown_counts.plot(kind="bar", ax=ax, color="steelblue")
ax.set_title('"unknown" values per column (hidden missing values)')
ax.set_ylabel("Rows")
for i, value in enumerate(unknown_counts):
    ax.text(i, value, f"{value:,}", ha="center", va="bottom")
plt.tight_layout()
plt.show()

rows_with_unknown = (df_raw[categorical_cols] == "unknown").any(axis=1).sum()
print(f"Rows with at least one unknown: {rows_with_unknown:,} ({rows_with_unknown / len(df_raw):.0%})")

In [ ]:
housing_unknown = df_raw["housing"] == "unknown"
loan_unknown = df_raw["loan"] == "unknown"
print("housing unknown:", housing_unknown.sum())
print("loan unknown:   ", loan_unknown.sum())
print("same rows?      ", (housing_unknown == loan_unknown).all())

**Decision: keep "unknown" as its own category.** There are no real NaNs; the missing values hide as the
word "unknown" in 6 columns. Dropping those rows would throw away 26% of the data, and filling them with
the most common value would invent information. `housing` and `loan` are unknown in exactly the same 990
rows, which points to a recording gap, so "unknown" itself may carry signal. One-hot encoding later gives
it its own column.

In [ ]:
df_raw["default"].value_counts()

`default` has only **3** "yes" values against 8,597 "unknown". The "yes" category is too rare to learn from,
but "unknown" vs "no" might still matter. We decide whether to keep this column in feature selection.

### 4.2 Duplicates

In [ ]:
duplicates = df_raw[df_raw.duplicated(keep=False)]
print("Duplicate rows (extra copies):", df_raw.duplicated().sum())
duplicates.sort_values(list(df_raw.columns)).head(6)

**Decision: drop the 12 extra copies.** The dataset has no client ID, so two identical rows cannot be told
apart and count as the same record twice.

### 4.3 Code values and inconsistent data

In [ ]:
never_contacted = df_raw["pdays"] == 999
print(f"pdays = 999 (never contacted): {never_contacted.mean():.1%} of rows")
print("Real pdays values range:", df_raw.loc[~never_contacted, "pdays"].min(), "to", df_raw.loc[~never_contacted, "pdays"].max())

contradiction = (df_raw["previous"] > 0) & never_contacted
print("Rows with previous > 0 but pdays = 999:", contradiction.sum())
pd.crosstab(df_raw["poutcome"], never_contacted.rename("pdays = 999"))

**Decision: replace `pdays` with a 0/1 `previously_contacted` flag.** 999 is a code meaning "never
contacted", not 999 days, so using it as a number would distort scaling. The flag step lives in the
shared `bankml` package so the deployed model applies it too.

**Inconsistency:** 4,110 rows say the client was contacted before (`previous` > 0, `poutcome` = failure
or success) while `pdays` says never. `previous` and `poutcome` agree with each other, so we trust them for
contact history, which is one more reason not to use raw `pdays`.

### 4.4 Outliers

In [ ]:
numeric_cols = list(df_raw.select_dtypes(include="number").columns)

outlier_rows = []
for col in numeric_cols:
    q1 = df_raw[col].quantile(0.25)
    q3 = df_raw[col].quantile(0.75)
    iqr = q3 - q1
    is_outlier = (df_raw[col] < q1 - 1.5 * iqr) | (df_raw[col] > q3 + 1.5 * iqr)
    outlier_rows.append({
        "column": col,
        "outliers": int(is_outlier.sum()),
        "min": df_raw[col].min(),
        "max": df_raw[col].max(),
    })

outliers = pd.DataFrame(outlier_rows).sort_values("outliers", ascending=False)
outliers

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(14, 4))
for ax, col in zip(axes, ["duration", "campaign", "age", "cons.conf.idx"]):
    sns.boxplot(y=df_raw[col], ax=ax, color="lightsteelblue")
    ax.set_title(col)
plt.suptitle("Columns with real outliers (IQR rule)")
plt.tight_layout()
plt.show()

**Decision: keep the outliers.** They are real values, not errors: a 98-year-old client, a client called 56
times, a very long call. The flags on `pdays` and `previous` are not real outliers either; those columns
are mostly one value (999 and 0), so any other value gets flagged. To soften the long tail of `campaign`,
the pipeline applies a log transform (`log1p`) before scaling.

### 4.5 Target balance

In [ ]:
target_share = df_raw["y"].value_counts(normalize=True)

fig, ax = plt.subplots(figsize=(5, 4))
sns.countplot(x=df_raw["y"], order=["no", "yes"], ax=ax, palette=["lightgray", "seagreen"], hue=df_raw["y"], legend=False)
ax.set_title("Did the client subscribe?")
for i, label in enumerate(["no", "yes"]):
    ax.text(i, (df_raw["y"] == label).sum(), f"{target_share[label]:.1%}", ha="center", va="bottom")
plt.tight_layout()
plt.show()

**The classes are imbalanced: 88.7% no, 11.3% yes.** A model that always says "no" would score 88.7%
accuracy and be useless, so we judge models mainly on **F1 for the "yes" class**, split the data with
`stratify` to keep the same balance in train and test, and use `class_weight="balanced"` where a model
supports it.

### 4.6 Clean and split

In [ ]:
df_clean = (
    df_raw
    .drop_duplicates()
    .reset_index(drop=True)
)
df_clean["y"] = (df_clean["y"] == "yes").astype(int)

X_full = df_clean.drop(columns=["y"])
y = df_clean["y"]

# stratify keeps the 11.3% "yes" share the same in both parts
X_train_full, X_test_full, y_train, y_test = train_test_split(
    X_full, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)
print("Train:", X_train_full.shape, " Test:", X_test_full.shape)

In [ ]:
assert len(df_clean) == 41176
assert abs(y_train.mean() - 0.113) < 0.002
assert abs(y_test.mean() - 0.113) < 0.002
print(f"Yes share: train {y_train.mean():.3f}, test {y_test.mean():.3f}")

### 4.7 Leakage check: the `duration` column

In [ ]:
categorical = list(X_train_full.select_dtypes(exclude="number").columns)
numeric_without = ["age", "previous", "emp.var.rate", "cons.price.idx", "cons.conf.idx",
                   "euribor3m", "nr.employed", "previously_contacted"]
numeric_with = numeric_without + ["duration"]

auc_scores = {}
for label, numeric in [("with duration", numeric_with), ("without duration", numeric_without)]:
    model = Pipeline([
        ("prep", build_preprocessor(categorical, numeric, ["campaign"])),
        ("model", LogisticRegression(max_iter=2000, class_weight="balanced")),
    ])
    scores = cross_val_score(model, X_train_full, y_train, cv=CV, scoring="roc_auc")
    auc_scores[label] = scores.mean()

pd.Series(auc_scores, name="mean CV ROC-AUC").round(3)

**Decision: drop `duration`.** With it the model looks far better, but call length is only known
**after** the call, and by then we already know the answer. A model that needs it cannot be used for
deciding who to call. This is called **data leakage**: information from the outcome sneaking into the
inputs.

In [ ]:
X_train = X_train_full.drop(columns=["duration"])
X_test = X_test_full.drop(columns=["duration"])
assert "duration" not in X_train.columns and "duration" not in X_test.columns
print("Final training inputs:", X_train.shape[1], "columns")

### 4.8 Relationships with the target (training data only)

In [ ]:
train = X_train.assign(y=y_train)

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
for ax, col in zip(axes.flat, ["job", "month", "poutcome", "contact"]):
    rate = (
        train
        .groupby(col)["y"]
        .mean()
        .sort_values(ascending=False)
    )
    sns.barplot(x=rate.index, y=rate.values, ax=ax, color="seagreen")
    ax.axhline(y_train.mean(), color="gray", linestyle="--", label="overall rate")
    ax.set_title(f"Subscription rate by {col}")
    ax.set_ylabel("Share who subscribed")
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=45)
    ax.legend()
plt.tight_layout()
plt.show()

A previous campaign **success** is by far the strongest signal (66% subscribe, against 11% overall).
Students (32%) and retirees (26%) subscribe most often, mobile contacts beat landlines (15% vs 5%), and
Mar, Sep, Oct and Dec reach 44 to 50%, while May, the busiest month, is only 6%.

In [ ]:
numeric_train = train.select_dtypes(include="number")

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(numeric_train.corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax)
ax.set_title("Correlation between numeric columns (training data)")
plt.tight_layout()
plt.show()

In [ ]:
euribor_bands = pd.cut(train["euribor3m"], bins=[0, 1, 2, 3, 4, 4.5, 5.1])
rate_by_band = train.groupby(euribor_bands, observed=True)["y"].mean()

fig, ax = plt.subplots(figsize=(8, 4))
rate_by_band.plot(kind="bar", ax=ax, color="steelblue")
ax.set_title("Subscription rate by 3-month Euribor rate")
ax.set_xlabel("euribor3m band (%)")
ax.set_ylabel("Share who subscribed")
ax.tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.show()

The economic columns move together strongly (`emp.var.rate`, `euribor3m` and `nr.employed` all above 0.9),
which we handle in feature selection. Low interest rates go with much higher subscription rates (45% when
Euribor was below 1%, about 5% above 4%): these calls span the 2008 financial crisis, when rates fell and
saving behaviour changed.